
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Task 3 - Feature Importance Analysis

In this task, we will perform feature importance analysis on the loan dataset. This analysis helps us understand which features have the most significant influence on predicting the target variable (e.g., Personal Loan). Feature importance analysis is useful for feature selection, model interpretability, and refining the dataset for model training.

**Objectives:**

- Train a Random Forest model to analyze feature importance.
- Identify the top features contributing to the model's predictions.
- Save a report with the most important features.

## Requirements

Please review the following requirements before starting the lesson:

* To run this notebook, you need to use one of the following Databricks runtime(s): **17.3.x-cpu-ml-scala2.13**

## Classroom Setup

Before starting the demo, run the provided classroom setup script. This script will define configuration variables necessary for the demo. Execute the following cell:

In [0]:
%run ../../Includes/Classroom-Setup-1.1demo

**Other Conventions:**

Throughout this demo, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [0]:
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")
print(f"Dataset Location:  {DA.paths.datasets}")


## Task Outline:
In this task, we will:

- Load the loan dataset.
- Preprocess the data by encoding categorical variables and preparing the feature set.
- Train a Random Forest model to assess feature importance.
- Identify and save the top important features.


###Step 1: Load Loan Data
In this step, we load the loan dataset from a Delta table and display a sample to verify the data structure.

In [0]:
# Define the dataset path
dataset_path = f"{DA.paths.datasets.banking}/banking/loan-clean.csv"

# Load the loan dataset
data = spark.read.format('csv').option('header', 'true').load(dataset_path)

# Display the first few rows to inspect the data
display(data)


###Step 2: Preprocess the Data
For this feature importance analysis, we will prepare the dataset by separating features and the target variable. We'll also perform encoding on categorical features.

**Instructions:**

- Drop any unnecessary columns (e.g., ID and ZIP_Code).
- Define Personal Loan as the target variable and the remaining columns as features.
- Convert categorical variables to numeric format for compatibility with the Random Forest model.

In [0]:
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer
from pyspark.sql.types import StringType
from sklearn.model_selection import train_test_split

# Define target and features
target = "Personal Loan"
X = data.drop("ID", "ZIP Code", target)  # Drop target and unnecessary columns
y = data.select(col(target).cast("int")).toPandas()[target]  # Convert target to integer and collect as pandas Series

# Encode categorical variables
categorical_cols = [field.name for field in X.schema.fields if isinstance(field.dataType, StringType)]

for col_name in categorical_cols:
    indexer = StringIndexer(inputCol=col_name, outputCol=col_name + "_indexed")
    X = indexer.fit(X).transform(X).drop(col_name)

# Convert X to pandas DataFrame for train_test_split
X = X.toPandas()

# Split the dataset for training
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

###Step 3: Train a Random Forest Model for Feature Importance
Using the Random Forest model, we can determine the importance of each feature in predicting the target variable.

**Instructions:**

- Train a Random Forest model on the training data.
- Extract feature importances and identify the top features.

In [0]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

# Initialize and train the model
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Get feature importances
feature_importances = pd.Series(model.feature_importances_, index=X.columns)
important_features = feature_importances.nlargest(5)

print("Top 5 Important Features")
print("========================")
print(important_features)


###Step 4: Save Feature Importance Report
We will save the top features identified by the model to a CSV file for reference and further analysis in the MLOps pipeline.

In [0]:
import json

# Convert important features to DataFrame
important_features_df = important_features.reset_index()
important_features_df.columns = ["Feature", "Importance"]

# Convert the sample display data to JSON format for API access and inspection
output_data = important_features_df.head(5).to_dict(orient="records")

# Print the JSON-formatted output for the final task
print("Notebook_Output:", json.dumps(output_data, indent=4))

# Save the visualization data and sample output to a JSON file
output_json_path = "./feature_engineered_output_with_visualization_data.json"
with open(output_json_path, "w") as json_file:
    json.dump({"sample_data": output_data}, json_file, indent=4)

print(f"JSON output saved to: {output_json_path}")

## Conclusion
In this notebook, we:

- Loaded and preprocessed the loan dataset.
- Trained a Random Forest model to analyze feature importance.
- Identified and saved the top features influencing the Personal Loan target variable.

The feature importance analysis provides insights into which features most impact the model's predictions, enabling better model interpretability and guiding feature selection for future modeling steps. The saved report will be used in subsequent steps of the pipeline for further validation and monitoring.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>